# ASOS Weather Observations: Usage Examples

This notebook demonstrates common access patterns for the ASOS surface weather observation dataset. The data is stored as partitioned GeoParquet files optimized for analytical queries.

## Prerequisites

```bash
pip install duckdb pandas geopandas matplotlib python-dotenv shapely
```

You'll also need R2 credentials in a `.env` file:
```
R2_ACCOUNT_ID=your_account_id
R2_ACCESS_KEY_ID=your_access_key
R2_SECRET_ACCESS_KEY=your_secret_key
```

In [ ]:
import os
from datetime import datetime

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from dotenv import load_dotenv

# Load credentials
load_dotenv()

# Configure plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
def get_connection():
    """Create a DuckDB connection configured for R2 access."""
    conn = duckdb.connect()
    
    account_id = os.environ.get("R2_ACCOUNT_ID")
    access_key = os.environ.get("R2_ACCESS_KEY_ID")
    secret_key = os.environ.get("R2_SECRET_ACCESS_KEY")
    
    conn.execute("INSTALL httpfs; LOAD httpfs;")
    conn.execute(f"SET s3_endpoint = '{account_id}.r2.cloudflarestorage.com';")
    conn.execute("SET s3_use_ssl = true;")
    conn.execute(f"SET s3_access_key_id = '{access_key}';")
    conn.execute(f"SET s3_secret_access_key = '{secret_key}';")
    conn.execute("SET s3_region = 'auto';")
    conn.execute("SET s3_url_style = 'path';")
    
    return conn

# Create connection
conn = get_connection()
print("Connection established")

## Data Access Patterns

The dataset uses yearly Hive-style partitioning. Choose the right access pattern based on your query:

### Single Year (Recommended for most queries)
```python
# Fast - directly accesses one file
read_parquet('s3://dev/asos/year=2015/data.parquet')
```

### Multiple Specific Years
```python
# Explicit list - no glob overhead
read_parquet([
    's3://dev/asos/year=2014/data.parquet',
    's3://dev/asos/year=2015/data.parquet'
])
```

### Multi-Year Range (use sparingly)
```python
# Glob pattern - scans all partitions first, then filters
# Only use when you genuinely need many years (climate normals, trends)
read_parquet('s3://dev/asos/year=*/data.parquet', hive_partitioning=true)
WHERE year BETWEEN 2010 AND 2015
```

**Why avoid glob when possible:**
- Glob must list and inspect all matching files before executing
- Higher memory overhead tracking partition metadata
- One corrupt partition can fail the whole query
- Browser (DuckDB-WASM) struggles with large glob patterns

This notebook uses direct file access for single-year queries and reserves glob patterns for genuinely multi-year analyses like climate normals.

---

## 1. Basic Data Exploration

Start by understanding the dataset structure, available years, and station coverage.

In [ ]:
# Check available years and record counts
year_summary = conn.execute(f"""
    SELECT 
        year,
        COUNT(*) as observations,
        COUNT(DISTINCT station) as stations,
        MIN(valid) as first_obs,
        MAX(valid) as last_obs
    FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
    GROUP BY year
    ORDER BY year DESC
    LIMIT 20
""").fetchdf()

print("Recent years in dataset:")
year_summary

In [ ]:
# Explore the schema
schema = conn.execute(f"""
    DESCRIBE SELECT * FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true) LIMIT 1
""").fetchdf()

print("Dataset schema:")
schema

In [ ]:
# Sample some data
sample = conn.execute(f"""
    SELECT station, state, valid, tmpf, dwpf, relh, sknt, vsby
    FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
    WHERE tmpf IS NOT NULL
    ORDER BY valid DESC
    LIMIT 10
""").fetchdf()

print("Sample observations:")
sample

---

## 2. Single Station Historical Analysis

Analyze temperature trends at a specific station over time. This example looks at Chicago O'Hare (KORD).

In [ ]:
# Query daily temperature statistics for Chicago O'Hare
station = "KORD"  # Chicago O'Hare
start_year = 2010
end_year = 2015

daily_temps = conn.execute(f"""
    SELECT 
        DATE_TRUNC('day', valid) as date,
        MIN(tmpf) as min_temp,
        AVG(tmpf) as avg_temp,
        MAX(tmpf) as max_temp,
        COUNT(*) as observations
    FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
    WHERE station = '{station}'
      AND year BETWEEN {start_year} AND {end_year}
      AND tmpf IS NOT NULL
    GROUP BY DATE_TRUNC('day', valid)
    ORDER BY date
""").fetchdf()

print(f"Retrieved {len(daily_temps):,} days of data for {station}")
daily_temps.head()

In [ ]:
# Plot daily temperature range
fig, ax = plt.subplots(figsize=(14, 6))

ax.fill_between(daily_temps['date'], daily_temps['min_temp'], daily_temps['max_temp'],
                alpha=0.3, color='steelblue', label='Daily Range')
ax.plot(daily_temps['date'], daily_temps['avg_temp'], color='steelblue', 
        linewidth=0.5, label='Daily Mean')

ax.set_xlabel('Date')
ax.set_ylabel('Temperature (°F)')
ax.set_title(f'Daily Temperature at {station} ({start_year}-{end_year})')
ax.legend()
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

In [ ]:
# Monthly climatology
monthly_climo = conn.execute(f"""
    SELECT 
        EXTRACT(MONTH FROM valid) as month,
        AVG(tmpf) as avg_temp,
        STDDEV(tmpf) as std_temp,
        MIN(tmpf) as record_low,
        MAX(tmpf) as record_high
    FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
    WHERE station = '{station}'
      AND year BETWEEN {start_year} AND {end_year}
      AND tmpf IS NOT NULL
    GROUP BY EXTRACT(MONTH FROM valid)
    ORDER BY month
""").fetchdf()

# Plot monthly climatology
months = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
          'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, ax = plt.subplots(figsize=(10, 6))
x = range(12)

ax.bar(x, monthly_climo['record_high'] - monthly_climo['record_low'], 
       bottom=monthly_climo['record_low'], alpha=0.3, color='steelblue',
       label='Record Range')
ax.errorbar(x, monthly_climo['avg_temp'], yerr=monthly_climo['std_temp'],
            fmt='o-', color='darkblue', capsize=3, label='Mean ± Std Dev')

ax.set_xticks(x)
ax.set_xticklabels(months)
ax.set_ylabel('Temperature (°F)')
ax.set_title(f'Monthly Temperature Climatology at {station}')
ax.legend()

plt.tight_layout()
plt.show()

---

## 3. Multi-Station Regional Comparison

Compare weather patterns across multiple stations in a region.

In [ ]:
# Compare major California airports
ca_stations = ['KLAX', 'KSFO', 'KSAN', 'KOAK', 'KSJC']
station_list = ", ".join(f"'{s}'" for s in ca_stations)

ca_monthly = conn.execute(f"""
    SELECT 
        station,
        EXTRACT(MONTH FROM valid) as month,
        AVG(tmpf) as avg_temp,
        AVG(relh) as avg_humidity
    FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
    WHERE station IN ({station_list})
      AND tmpf IS NOT NULL
    GROUP BY station, EXTRACT(MONTH FROM valid)
    ORDER BY station, month
""").fetchdf()

# Pivot for plotting
temp_pivot = ca_monthly.pivot(index='month', columns='station', values='avg_temp')

fig, ax = plt.subplots(figsize=(12, 6))
temp_pivot.plot(ax=ax, marker='o')

ax.set_xticks(range(1, 13))
ax.set_xticklabels(months)
ax.set_xlabel('Month')
ax.set_ylabel('Average Temperature (°F)')
ax.set_title('Monthly Temperature Comparison: California Airports (2015)')
ax.legend(title='Station')

plt.tight_layout()
plt.show()

In [ ]:
# Diurnal cycle comparison
diurnal = conn.execute(f"""
    SELECT 
        station,
        EXTRACT(HOUR FROM valid) as hour,
        AVG(tmpf) as avg_temp
    FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
    WHERE station IN ({station_list})
      AND EXTRACT(MONTH FROM valid) = 7  -- July only
      AND tmpf IS NOT NULL
    GROUP BY station, EXTRACT(HOUR FROM valid)
    ORDER BY station, hour
""").fetchdf()

diurnal_pivot = diurnal.pivot(index='hour', columns='station', values='avg_temp')

fig, ax = plt.subplots(figsize=(12, 6))
diurnal_pivot.plot(ax=ax, marker='o')

ax.set_xlabel('Hour (UTC)')
ax.set_ylabel('Average Temperature (°F)')
ax.set_title('Diurnal Temperature Cycle: California Airports (July 2015)')
ax.legend(title='Station')
ax.set_xticks(range(0, 24, 3))

plt.tight_layout()
plt.show()

---

## 4. Extreme Weather Event Detection

Find heat waves, cold snaps, and other extreme weather events.

In [ ]:
# Find the hottest days on record (with data quality filtering)
hottest_days = conn.execute(f"""
    WITH daily_max AS (
        SELECT 
            station,
            state,
            DATE_TRUNC('day', valid) as date,
            MAX(tmpf) as max_temp,
            -- Get dewpoint at time of max temp for validation
            FIRST(dwpf ORDER BY tmpf DESC) as dewpoint_at_max
        FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
        WHERE year BETWEEN 2000 AND 2015
          AND tmpf IS NOT NULL
          AND tmpf BETWEEN 80 AND 135  -- Reasonable range
          -- Data quality: require valid dewpoint
          AND dwpf IS NOT NULL
          AND dwpf <= tmpf
          AND dwpf >= 0
        GROUP BY station, state, DATE_TRUNC('day', valid)
    )
    SELECT *
    FROM daily_max
    WHERE max_temp >= 115  -- Extreme heat threshold
    ORDER BY max_temp DESC
    LIMIT 20
""").fetchdf()

print("Hottest days (115°F+) with validated dewpoints:")
hottest_days

In [ ]:
# Find extreme cold events in the northern US
coldest_days = conn.execute(f"""
    WITH daily_min AS (
        SELECT 
            station,
            state,
            DATE_TRUNC('day', valid) as date,
            MIN(tmpf) as min_temp
        FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
        WHERE year BETWEEN 2000 AND 2015
          AND state IN ('MN', 'ND', 'MT', 'WI', 'MI')
          AND tmpf IS NOT NULL
          AND tmpf BETWEEN -60 AND 32
          -- Validate with dewpoint
          AND dwpf IS NOT NULL
          AND dwpf <= tmpf
        GROUP BY station, state, DATE_TRUNC('day', valid)
    )
    SELECT *
    FROM daily_min
    WHERE min_temp <= -30
    ORDER BY min_temp ASC
    LIMIT 20
""").fetchdf()

print("Coldest days (-30°F or below) in northern states:")
coldest_days

In [ ]:
# Detect heat waves (3+ consecutive days above 100°F)
heat_wave_query = conn.execute(f"""
    WITH daily_max AS (
        SELECT 
            station,
            state,
            DATE_TRUNC('day', valid) as date,
            MAX(tmpf) as max_temp
        FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
        WHERE tmpf IS NOT NULL
          AND dwpf IS NOT NULL
          AND dwpf <= tmpf
        GROUP BY station, state, DATE_TRUNC('day', valid)
    ),
    hot_days AS (
        SELECT *,
            CASE WHEN max_temp >= 100 THEN 1 ELSE 0 END as is_hot
        FROM daily_max
    ),
    streaks AS (
        SELECT *,
            SUM(CASE WHEN is_hot = 0 THEN 1 ELSE 0 END) 
                OVER (PARTITION BY station ORDER BY date) as streak_group
        FROM hot_days
    )
    SELECT 
        station,
        state,
        MIN(date) as start_date,
        MAX(date) as end_date,
        COUNT(*) as consecutive_days,
        MAX(max_temp) as peak_temp
    FROM streaks
    WHERE is_hot = 1
    GROUP BY station, state, streak_group
    HAVING COUNT(*) >= 3
    ORDER BY consecutive_days DESC, peak_temp DESC
    LIMIT 15
""").fetchdf()

print("Heat waves (3+ consecutive days >= 100°F) in 2015:")
heat_wave_query

---

## 5. Spatial Interpolation

Estimate weather conditions at locations between stations using Inverse Distance Weighting (IDW).

In [ ]:
def haversine_distance(lat1, lon1, lat2, lon2):
    """Calculate distance in km between two points."""
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    return 6371 * 2 * np.arcsin(np.sqrt(a))

def idw_interpolate(target_lat, target_lon, lats, lons, values, power=2):
    """Inverse Distance Weighting interpolation."""
    distances = haversine_distance(target_lat, target_lon, 
                                   np.array(lats), np.array(lons))
    
    # Handle exact match
    if np.any(distances < 0.01):
        return values[np.argmin(distances)]
    
    weights = 1.0 / (distances ** power)
    weights /= weights.sum()
    return np.sum(weights * np.array(values))

In [ ]:
# Target location: A solar farm in Tennessee
target_lat, target_lon = 35.3475, -89.8625
target_name = "Tennessee Solar Farm"

# Find nearest stations
nearest_stations = conn.execute(f"""
    WITH station_locs AS (
        SELECT DISTINCT
            station,
            FIRST(latitude) as lat,
            FIRST(longitude) as lon
        FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
        WHERE latitude IS NOT NULL
        GROUP BY station
    )
    SELECT 
        station, lat, lon,
        6371 * 2 * ASIN(SQRT(
            POWER(SIN(RADIANS(lat - {target_lat}) / 2), 2) +
            COS(RADIANS({target_lat})) * COS(RADIANS(lat)) *
            POWER(SIN(RADIANS(lon - {target_lon}) / 2), 2)
        )) as distance_km
    FROM station_locs
    ORDER BY distance_km
    LIMIT 5
""").fetchdf()

print(f"Nearest stations to {target_name}:")
nearest_stations

In [ ]:
# Get hourly data for these stations on a specific day
station_list = ", ".join(f"'{s}'" for s in nearest_stations['station'])
query_date = "2015-07-15"

hourly_data = conn.execute(f"""
    SELECT 
        station,
        valid,
        latitude,
        longitude,
        tmpf
    FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
    WHERE station IN ({station_list})
      AND DATE_TRUNC('day', valid) = DATE '{query_date}'
      AND tmpf IS NOT NULL
    ORDER BY valid, station
""").fetchdf()

# Interpolate for each hour
interpolated = []
for timestamp, group in hourly_data.groupby('valid'):
    if len(group) >= 2:
        temp = idw_interpolate(
            target_lat, target_lon,
            group['latitude'].values,
            group['longitude'].values,
            group['tmpf'].values
        )
        interpolated.append({'timestamp': timestamp, 'temp_f': temp})

interp_df = pd.DataFrame(interpolated)

# Plot interpolated vs station data
fig, ax = plt.subplots(figsize=(12, 6))

# Plot each station
for station in nearest_stations['station']:
    stn_data = hourly_data[hourly_data['station'] == station]
    ax.plot(stn_data['valid'], stn_data['tmpf'], 'o-', alpha=0.5, 
            label=f"{station} ({nearest_stations[nearest_stations['station']==station]['distance_km'].values[0]:.1f} km)")

# Plot interpolated
ax.plot(interp_df['timestamp'], interp_df['temp_f'], 'k-', linewidth=2,
        label=f'Interpolated at {target_name}')

ax.set_xlabel('Time (UTC)')
ax.set_ylabel('Temperature (°F)')
ax.set_title(f'IDW Interpolation at {target_name} ({query_date})')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))

plt.tight_layout()
plt.show()

---

## 6. Climate Normals Computation

Calculate 30-year climatological normals for a station.

In [ ]:
# Compute climate normals for Denver (1985-2014 period)
station = "KDEN"  # Denver International

climate_normals = conn.execute(f"""
    WITH daily_stats AS (
        SELECT 
            EXTRACT(MONTH FROM valid) as month,
            EXTRACT(DAY FROM valid) as day,
            EXTRACT(YEAR FROM valid) as year,
            AVG(tmpf) as daily_avg_temp,
            MIN(tmpf) as daily_min_temp,
            MAX(tmpf) as daily_max_temp,
            SUM(COALESCE(p01i, 0)) as daily_precip
        FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
        WHERE station = '{station}'
          AND year BETWEEN 1985 AND 2014
          AND tmpf IS NOT NULL
        GROUP BY EXTRACT(MONTH FROM valid), EXTRACT(DAY FROM valid), EXTRACT(YEAR FROM valid)
    )
    SELECT 
        month,
        day,
        AVG(daily_avg_temp) as normal_temp,
        AVG(daily_min_temp) as normal_min,
        AVG(daily_max_temp) as normal_max,
        AVG(daily_precip) as normal_precip,
        COUNT(DISTINCT year) as years_of_data
    FROM daily_stats
    GROUP BY month, day
    ORDER BY month, day
""").fetchdf()

# Create day-of-year index
climate_normals['doy'] = range(1, len(climate_normals) + 1)

print(f"Climate normals for {station} (1985-2014):")
climate_normals.head(10)

In [ ]:
# Plot climate normals
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Temperature normals
ax1.fill_between(climate_normals['doy'], 
                 climate_normals['normal_min'],
                 climate_normals['normal_max'],
                 alpha=0.3, color='steelblue', label='Normal Min-Max Range')
ax1.plot(climate_normals['doy'], climate_normals['normal_temp'], 
         color='darkblue', linewidth=2, label='Normal Mean')

ax1.set_xlabel('Day of Year')
ax1.set_ylabel('Temperature (°F)')
ax1.set_title(f'Temperature Normals at {station} (1985-2014)')
ax1.legend()
ax1.set_xlim(1, 365)

# Monthly precipitation
monthly_precip = climate_normals.groupby('month')['normal_precip'].sum()
ax2.bar(range(1, 13), monthly_precip.values, color='steelblue', alpha=0.7)
ax2.set_xticks(range(1, 13))
ax2.set_xticklabels(months)
ax2.set_xlabel('Month')
ax2.set_ylabel('Normal Precipitation (inches)')
ax2.set_title(f'Monthly Precipitation Normals at {station}')

plt.tight_layout()
plt.show()

---

## 7. Data Quality Assessment

Identify and filter potentially erroneous observations.

In [ ]:
# Check for suspicious "round Celsius" values
# These often indicate sensor calibration errors
suspicious_temps = [134.6, 132.8, 131.0, 129.2, -56.2, -58.0, -59.8]
temp_list = ", ".join(str(t) for t in suspicious_temps)

suspicious = conn.execute(f"""
    SELECT 
        station,
        valid,
        tmpf,
        ROUND((tmpf - 32) * 5/9, 1) as temp_celsius,
        dwpf,
        relh
    FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
    WHERE year BETWEEN 2000 AND 2010
      AND tmpf IN ({temp_list})
    ORDER BY tmpf DESC
    LIMIT 20
""").fetchdf()

print("Suspicious readings (round Celsius values converted to Fahrenheit):")
suspicious

In [ ]:
# Find stations with highest rates of suspicious data
station_quality = conn.execute(f"""
    WITH all_obs AS (
        SELECT station, COUNT(*) as total
        FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
        WHERE year BETWEEN 2000 AND 2010
          AND tmpf IS NOT NULL
        GROUP BY station
    ),
    suspicious_obs AS (
        SELECT station, COUNT(*) as suspicious
        FROM read_parquet('{DATA_SOURCE}', hive_partitioning=true)
        WHERE year BETWEEN 2000 AND 2010
          AND tmpf IN ({temp_list})
        GROUP BY station
    )
    SELECT 
        a.station,
        a.total,
        COALESCE(s.suspicious, 0) as suspicious,
        ROUND(COALESCE(s.suspicious, 0) * 100.0 / a.total, 4) as pct_suspicious
    FROM all_obs a
    LEFT JOIN suspicious_obs s ON a.station = s.station
    WHERE s.suspicious > 0
    ORDER BY pct_suspicious DESC
    LIMIT 15
""").fetchdf()

print("Stations with highest rates of suspicious readings:")
station_quality

In [ ]:
# Recommended data quality filters
print("""
=== Recommended Data Quality Filters ===

For temperature analysis:
```sql
WHERE tmpf IS NOT NULL
  AND dwpf IS NOT NULL           -- Require dewpoint for cross-validation
  AND dwpf <= tmpf               -- Dewpoint cannot exceed temperature
  AND dwpf >= -40                -- Reasonable dewpoint floor
  AND tmpf BETWEEN -60 AND 130   -- Physical limits for continental US
```

For precipitation:
```sql
WHERE p01i IS NOT NULL
  AND p01i >= 0
  AND p01i < 5  -- Filter > 5 inches/hour as likely errors
```

For wind:
```sql
WHERE sknt IS NOT NULL
  AND sknt >= 0
  AND sknt < 150  -- Filter > 150 knots as likely errors
  AND (drct IS NULL OR drct BETWEEN 0 AND 360)
```
""")

---

## 8. Wind Rose Analysis

Analyze wind patterns at a specific station.

In [ ]:
# Get wind data for a coastal station
station = "KSFO"  # San Francisco

wind_data = conn.execute(f"""
    SELECT 
        drct as direction,
        sknt as speed
    FROM read_parquet('s3://dev/asos/year=2015/data.parquet')
    WHERE station = '{station}'
      AND sknt IS NOT NULL
      AND sknt > 0
      AND drct IS NOT NULL
      AND drct BETWEEN 0 AND 360
""").fetchdf()

print(f"Wind observations at {station}: {len(wind_data):,}")

# Bin into 16 direction sectors
wind_data['sector'] = ((wind_data['direction'] + 11.25) // 22.5).astype(int) % 16

# Speed categories
speed_bins = [0, 5, 10, 15, 20, 100]
speed_labels = ['0-5', '5-10', '10-15', '15-20', '20+']
wind_data['speed_cat'] = pd.cut(wind_data['speed'], bins=speed_bins, labels=speed_labels)

# Create wind rose data
wind_rose = wind_data.groupby(['sector', 'speed_cat']).size().unstack(fill_value=0)
wind_rose = wind_rose / len(wind_data) * 100  # Convert to percentages

print("\nWind frequency by direction and speed (%):\n")
wind_rose

In [ ]:
# Plot wind rose
directions = ['N', 'NNE', 'NE', 'ENE', 'E', 'ESE', 'SE', 'SSE',
              'S', 'SSW', 'SW', 'WSW', 'W', 'WNW', 'NW', 'NNW']

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

theta = np.linspace(0, 2*np.pi, 16, endpoint=False)
# Rotate so N is at top
theta = theta - np.pi/2

width = 2*np.pi / 16 * 0.8
colors = plt.cm.Blues(np.linspace(0.3, 1, len(speed_labels)))

bottom = np.zeros(16)
for i, speed_cat in enumerate(speed_labels):
    if speed_cat in wind_rose.columns:
        values = wind_rose[speed_cat].values
        ax.bar(theta, values, width=width, bottom=bottom, 
               color=colors[i], label=f'{speed_cat} kt')
        bottom += values

ax.set_theta_zero_location('N')
ax.set_theta_direction(-1)
ax.set_thetagrids(np.arange(0, 360, 22.5), directions)
ax.set_title(f'Wind Rose for {station} (2015)', y=1.08, fontsize=14)
ax.legend(loc='upper left', bbox_to_anchor=(1.1, 1))

plt.tight_layout()
plt.show()

---

## Summary

This notebook demonstrated:

1. **Basic Data Exploration** - Schema discovery, year coverage, sampling
2. **Single Station Analysis** - Historical temperature trends, monthly climatology
3. **Multi-Station Comparison** - Regional weather patterns, diurnal cycles
4. **Extreme Event Detection** - Heat waves, cold snaps with data quality filters
5. **Spatial Interpolation** - IDW estimation at locations between stations
6. **Climate Normals** - Computing 30-year climatological averages
7. **Data Quality Assessment** - Identifying and filtering suspect observations
8. **Wind Rose Analysis** - Directional wind pattern visualization

### Tips for Working with ASOS Data

- **Always filter by year** using Hive partitioning to limit data scanned
- **Cross-validate temperatures** with dewpoint when detecting extremes
- **Check station metadata** before assuming continuous coverage
- **Use UTC timestamps** - all observations are in UTC
- **Handle missing data** - gust and precipitation are sparse (only reported when events occur)